# Bayesian networks with pyAgrum

**The room.** The university gym has a weights room and, on the other side of one wall, a studio where
group classes are held. The weights room has three sensors: a microphone that reports when the sound
level goes over a threshold, a CO&#8322; sensor, and an accelerometer bolted to the floor structure.

Two things can be going on in there, and neither of them is measured: a **class** is running next door,
and the weights room is **busy**. The system has to work out which, from the three sensors alone.

One sensor is what makes this interesting. **The microphone is in the weights room, but it hears both rooms.**

Today you build the network that does this reasoning, put your own numbers into it, watch it change its
mind, and then throw your numbers away and count them out of two weeks of logs instead.

**Teaching dataset:** `gym_log.csv` is synthetic, generated for this exercise. The gym is an illustrative scenario. The columns play the roles of situation labels and sensor observations; they are not measurements collected in a real gym.


In [ ]:
#@title Run this cell first. It installs pyAgrum and defines the helpers — you never need to read it.
!pip install -q pyagrum

import os, itertools, urllib.request
import pandas as pd
import pyagrum as gum
import pyagrum.lib.notebook as gnb

NODES = ["Class", "Busy", "Sound", "CO2", "Vibration"]
LOG_URL = ("https://raw.githubusercontent.com/lucregrassi/ambient-intelligence-labs"
           "/main/lab4-bayesian-networks/gym_log.csv")

if not os.path.exists("gym_log.csv"):
    urllib.request.urlretrieve(LOG_URL, "gym_log.csv")
log = pd.read_csv("gym_log.csv")

def new_network():
    """Five binary nodes, no arcs yet."""
    bn = gum.BayesNet("The weights room")
    for n in NODES:
        bn.add(gum.LabelizedVariable(n, n, ["no", "yes"]))
    return bn

def parents_of(bn, node):
    return list(bn.cpt(node).names[1:])

def _condition(key):
    """'Class=yes' or ('Class=yes', 'Busy=no') -> {'Class': 'yes', ...}"""
    keys = (key,) if isinstance(key, str) else key
    return dict(k.split("=") for k in keys)

def fill(bn, P):
    """Put your numbers into the network. Every number is P(node = yes)."""
    bn.cpt("Class").fillWith([1 - P["Class"], P["Class"]])
    bn.cpt("Busy").fillWith([1 - P["Busy"], P["Busy"]])
    for node in ("Sound", "Vibration"):
        for key, v in P[node].items():
            bn.cpt(node)[_condition(key)] = [1 - v, v]
    for key, v in P["CO2"].items():
        cond = _condition(key)
        if "Class" in parents_of(bn, "CO2"):    # optional arc: the same guess on both sides
            for c in ("no", "yes"):
                bn.cpt("CO2")[dict(cond, **{"Class": c})] = [1 - v, v]
        else:
            bn.cpt("CO2")[cond] = [1 - v, v]

def show(bn, size="5"):
    gnb.showBN(bn, size=size)

def show_cpts(bn):
    gnb.sideBySide(*[bn.cpt(n) for n in NODES], captions=NODES)

def ask(bn, evidence=None, size="8"):
    gnb.showInference(bn, evs=evidence or {}, size=size)

def belief(bn, evidence=None):
    ie = gum.LazyPropagation(bn)
    ie.setEvidence(evidence or {})
    ie.makeInference()
    return {n: round(100 * float(ie.posterior(n).toarray()[1]), 1) for n in ("Class", "Busy")}

def as_text(evidence):
    return ", ".join(f"{k}={v}" for k, v in evidence.items()) or "nothing observed"

def one_clue_at_a_time(bn, clues):
    """clues: a list of (node, value). Adds them one by one and reports the two situations."""
    rows, ev = [], {}
    rows.append({"what the system knows": as_text(ev), **belief(bn, ev)})
    for node, value in clues:
        ev = dict(ev, **{node: value})
        rows.append({"what the system knows": as_text(ev), **belief(bn, ev)})
    return pd.DataFrame(rows).set_index("what the system knows")

def learn(bn, n_cases):
    """Count the cases in the log and turn the counts into CPTs. Same structure, new numbers."""
    log.head(n_cases).to_csv("_subset.csv", index=False)
    learner = gum.BNLearner("_subset.csv", bn)
    learner.useSmoothingPrior(1)
    return learner.learnParameters(bn.dag())

def compare_cpt(bn_you, bn_log, node):
    parents = parents_of(bn_you, node)
    rows = []
    for combo in itertools.product(["no", "yes"], repeat=len(parents)):
        ev = dict(zip(parents, combo))
        row = {p: v for p, v in ev.items()}
        row[f"P({node}=yes) — you"] = round(float(bn_you.cpt(node)[ev][1]), 3)
        row[f"P({node}=yes) — the log"] = round(float(bn_log.cpt(node)[ev][1]), 3)
        rows.append(row)
    return pd.DataFrame(rows)

def compare_answers(bn_you, bn_log, evidence):
    return pd.DataFrame([{"": "your numbers", **belief(bn_you, evidence)},
                         {"": "numbers from the log", **belief(bn_log, evidence)}]).set_index("")

## 1. The structure

Before you run anything: on paper, in your group, draw the network. Five binary nodes &mdash;
`Class`, `Busy`, `Sound`, `CO2`, `Vibration` &mdash; and the arrows between them. For this exercise we choose arrows with a causal interpretation. A Bayesian network in general encodes a factorisation; its arrows do not automatically establish causality.

<details>
<summary><b>Draw it first, then open this</b></summary>

`Sound` has two parents, and that is the whole point of the lab: the microphone cannot tell which room
it heard. `CO2` has one parent, `Busy` &mdash; the studio has its own air handling. `Vibration` has one
parent, `Class`. And there is no arrow between `Class` and `Busy`: we are claiming that a class next
door does not change how busy the weights room is. That is an assumption about this building, not a
fact, and it is the first thing worth arguing about.
</details>

In [ ]:
bn = new_network()                  # five binary nodes, no arcs yet
bn.addArc("Class", "Sound")         # the class is heard through the wall
bn.addArc("Busy",  "Sound")         # a busy weights room is loud on its own
bn.addArc("Busy",  "CO2")           # the people in this room breathe
bn.addArc("Class", "Vibration")     # a jumping class shakes the floor
# bn.addArc("Class", "CO2")         # <- section 5 will tell you whether this one belongs here
show(bn)

`Sound` is a **collider**: two causes meeting at one effect. Everything surprising further down happens
at that node.

## 2. The numbers

The structure says *which* numbers the network needs. It does not say what they are. Here they are:
one for `Class`, one for `Busy`, four for `Sound` (one per combination of its two parents), two for
`CO2`, two for `Vibration`. **Ten numbers** &mdash; the full joint distribution over five binary
variables would have needed 31.

Nobody measured these. They are what a person who knows the building would say. Change the one marked
with an arrow if you disagree: it is the wall.

In [ ]:
P = {                                            # every number is P(the node = yes)
    "Class": 0.30,                               # a class runs about a third of opening hours
    "Busy":  0.40,

    "Sound": {
        ("Class=yes", "Busy=no"):  0.80,         # <- the wall: how much of the class comes through
        ("Class=yes", "Busy=yes"): 0.97,
        ("Class=no",  "Busy=yes"): 0.70,
        ("Class=no",  "Busy=no"):  0.05,         # the room is never completely silent
    },

    "CO2":       {"Busy=yes":  0.75, "Busy=no":  0.05},
    "Vibration": {"Class=yes": 0.65, "Class=no": 0.03},
}
fill(bn, P)
show_cpts(bn)

## 3. Asking it a question

The microphone goes over the threshold. Nothing else is known.

In [ ]:
ask(bn, {"Sound": "yes"})

The orange node is what you observed; the bars on the other four are what the network now believes.
Both situations went up, which is what you would expect: something made a noise.

## 4. One clue at a time

Now give it the other two sensors, one at a time, and watch the two situations move.

In [ ]:
one_clue_at_a_time(bn, [("Sound", "yes"), ("CO2", "no"), ("Vibration", "yes")])

Read the third row. Before conditioning on `Sound`, the model makes `CO2` and `Class` independent. Observing `Sound` opens the collider path `CO2 <- Busy -> Sound <- Class`. A normal CO2 reading then weakens the busy-room explanation, and `P(Class = yes)` rises from about 55% to 71%.

Now read the last row. `P(Busy = yes)` ends at **about 18.5%, below its 40% prior**, after observing sound, normal CO2 and vibration. The vibration supports a class; the class can explain the sound, reducing the need for the busy-room explanation. This is **explaining away** between competing parents of an observed collider.

<details>
<summary><b>Before you run the next cell: what happens if the floor is still instead?</b></summary>

Changing `Vibration` from `yes` to `no` weakens the class explanation. Busy rises to about 46.2%, while Class falls to about 46.5%. Neither explanation becomes certain or impossible: the CPTs also allow noisy observations. Try changing the evidence and explain the direction before reading the numbers.
</details>


## 5. Estimating the numbers from labelled examples

`gym_log.csv` contains 2016 **synthetic** examples generated for this exercise. The scenario presents these as two weeks of five-minute operating slots. The supplied file contains no timestamps and its rows are independent simulated cases, not a measured time series.

`Class` and `Busy` are the training labels; the last three columns play the role of sensor observations. In a real deployment these labels would need an independent reference and validation. Booking and turnstile records may be noisy proxies, and should not silently be treated as truth.


In [ ]:
log.head()

In [ ]:
N_CASES = 2016                 # try 50, then 200, then all 2016
bn_log = learn(bn, N_CASES)
compare_cpt(bn, bn_log, "Sound")

The wall is the number you got most wrong: you said 0.80 of the class gets through, the log says 0.55.
Other guesses also differ: compare all CPT rows. In particular CO2 given Busy and Vibration given Class deserve attention.

Now set `N_CASES = 50` and run that cell again. The same column moves by tens of points, because each
of those four numbers is being counted off a handful of rows. **A CPT estimated from few cases has high sampling uncertainty.** (The notebook adds one imaginary case of each kind
before counting, so that a combination nobody ever saw does not come out as a certainty.)

In [ ]:
compare_cpt(bn, bn_log, "CO2")

Two rows, because `CO2` has one parent. Go back to section 1, **uncomment the `Class -> CO2` arc**, and rerun sections 1, 2 and 5. The fitted table has four rows; the two `Busy = yes` rows are about 0.44 and 0.47.

The estimates are close, but a small observed difference does not prove conditional independence. Compare the sample sizes in each parent row and their uncertainty; assess whether the added arc improves held-out predictions. We know the teaching generator omits this dependence. A real deployment would need independent evidence for its own structure.


## 6. The same question, asked twice

Loud room, normal CO&#8322;, still floor. The most ordinary evening in the gym, and the one case where
the two networks do not agree.

In [ ]:
EVIDENCE = {"Sound": "yes", "CO2": "no", "Vibration": "no"}
compare_answers(bn, bn_log, EVIDENCE)

In [ ]:
ask(bn_log, EVIDENCE)

With your numbers the system shrugs: 46 and 46, it cannot choose between the two situations. With the
numbers counted from the log it does choose &mdash; no class, the room is busy, 69%.

Same five nodes, same four arrows, same question. **The structure decides which questions the network
can answer; the numbers decide what it answers when the evidence is ambiguous** &mdash; which, in a
real building, is most evenings.

## What to take away

1. Each parent changes the conditional distribution that must be specified. With two binary parents, `Sound` needs four parent configurations. Our exercise gives arrows a causal interpretation; a BN alone does not prove that interpretation.
2. Evidence can change beliefs in either direction along active paths. Observing a collider can make its otherwise independent parents dependent; this explains the changing Class and Busy beliefs.
3. Conditional counts, explicit expert priors and other justified estimation methods can fill CPTs. Report sparse rows, recognise sampling uncertainty and validate predictions on held-out data. These simulated examples illustrate the method; they do not validate a real gym system.
